In [6]:
import os
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
path = os.path.join( "..", "data", "processed", "accidents_clean.csv")
accidents_clean = pd.read_csv(path)

In [7]:
# Effectue un test de Chi2 + V de Cramer sur les variables catégorielles

target_variable = 'grav'
variables_exclues = ['id_vehicule', 'num_veh', 'hrmn','lat', 'long', 'age', 'date']

def cramers_v(chi2, n, min_dim):
    """Calculer le V de Cramer."""
    return np.sqrt(chi2 / (n * (min_dim - 1)))


for column in accidents_clean.columns:
    # Ignorer la variable cible elle-même
    if column == target_variable or column in variables_exclues:
        continue

    # Créer une table de contingence
    contingency_table = pd.crosstab(accidents_clean[target_variable], accidents_clean[column])

    # Effectuer le test du chi-carré
    chi2_result = chi2_contingency(contingency_table)

    # Calculer le V de Cramer
    n = contingency_table.sum().sum()
    min_dim = min(contingency_table.shape) - 1
    cramers_v_value = cramers_v(chi2_result[0], n, min_dim)

    # Afficher les résultats
    print(f"Effet de {column} sur la gravité : X2 = {chi2_result[0]}, p = {chi2_result[1]}, V de Cramer = {cramers_v_value}")

Effet de catv sur la gravité : X2 = 111324.76226585373, p = 0.0, V de Cramer = 0.30095329236735924
Effet de obs sur la gravité : X2 = 47038.071451360214, p = 0.0, V de Cramer = 0.19562662585510596
Effet de obsm sur la gravité : X2 = 38635.62586318465, p = 0.0, V de Cramer = 0.17729528516510806
Effet de choc sur la gravité : X2 = 27743.002097430995, p = 0.0, V de Cramer = 0.15023804995679885
Effet de manv sur la gravité : X2 = 48973.02071035564, p = 0.0, V de Cramer = 0.1996097068800241
Effet de motor sur la gravité : X2 = 18997.59528475653, p = 0.0, V de Cramer = 0.12432329912026802
Effet de place sur la gravité : X2 = 38608.70431524888, p = 0.0, V de Cramer = 0.17723350416406675
Effet de catu sur la gravité : X2 = 36835.10901923972, p = 0.0, V de Cramer = 0.24482129351582788
Effet de sexe sur la gravité : X2 = 5142.549087480826, p = 0.0, V de Cramer = inf
Effet de trajet sur la gravité : X2 = 22154.76417236431, p = 0.0, V de Cramer = 0.1342569498568274
Effet de secu1 sur la gravité : 

/var/folders/05/_vxzr9653lg058wd1tp1rdyr0000gn/T/ipykernel_7107/2667022449.py:8: RuntimeWarning: divide by zero encountered in scalar divide
  return np.sqrt(chi2 / (n * (min_dim - 1)))


Effet de an sur la gravité : X2 = 153.12297837204093, p = 1.3169146370967613e-26, V de Cramer = 0.01116152135589487
Effet de lum sur la gravité : X2 = 8592.599936043642, p = 0.0, V de Cramer = 0.08361140214944744
Effet de dep sur la gravité : X2 = 48336.817585433295, p = 0.0, V de Cramer = 0.19830891458508718
Effet de com sur la gravité : X2 = 220039.39466474735, p = 0.0, V de Cramer = 0.42311013791821195
Effet de agg sur la gravité : X2 = 17589.046204036513, p = 0.0, V de Cramer = inf
Effet de int sur la gravité : X2 = 5466.042525394254, p = 0.0, V de Cramer = 0.06668677475485614
Effet de atm sur la gravité : X2 = 1958.1603031291438, p = 0.0, V de Cramer = 0.0399141910389684
Effet de col sur la gravité : X2 = 42666.75562215519, p = 0.0, V de Cramer = 0.18631508577977576
Effet de catr sur la gravité : X2 = 21575.686810743035, p = 0.0, V de Cramer = 0.1324907398480452
Effet de circ sur la gravité : X2 = 12104.233519593441, p = 0.0, V de Cramer = 0.09923659569982872
Effet de nbv sur la g

/var/folders/05/_vxzr9653lg058wd1tp1rdyr0000gn/T/ipykernel_7107/2667022449.py:8: RuntimeWarning: divide by zero encountered in scalar divide
  return np.sqrt(chi2 / (n * (min_dim - 1)))


Effet de catv_regroupée sur la gravité : X2 = 102146.03761317201, p = 0.0, V de Cramer = 0.28827964116391624


In [18]:
# GLM avec distribution de Poisson pour tester l'effet de l'age sur la gravité de l'accident
# Important : Install statsmodels !


import statsmodels.api as sm
import statsmodels.formula.api as smf

glm_model = smf.glm(formula='grav ~ age',
                    data=accidents_clean,
                    family=sm.families.Poisson()).fit()

# Afficher le résumé du modèle
print(glm_model.summary())

coef = glm_model.params['age']
effet = np.exp(coef)
print(f"Effet multiplicatif de l'âge : {effet:.3f}")

                 Generalized Linear Model Regression Results                  
Dep. Variable:                   grav   No. Observations:               614559
Model:                            GLM   Df Residuals:                   614557
Model Family:                 Poisson   Df Model:                            1
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -8.5228e+05
Date:                Tue, 17 Jun 2025   Deviance:                   2.1078e+05
Time:                        17:55:50   Pearson chi2:                 2.19e+05
No. Iterations:                     4   Pseudo R-squ. (CS):          5.074e-05
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.5691      0.002    261.752      0.0

In [20]:
glm_model = smf.glm(formula='grav ~ nbv',
                    data=accidents_clean,
                    family=sm.families.Poisson()).fit()

# Afficher le résumé du modèle
print(glm_model.summary())

coef = glm_model.params['nbv']
effet = np.exp(coef)
print(f"Effet multiplicatif de nbv : {effet:.3f}")

                 Generalized Linear Model Regression Results                  
Dep. Variable:                   grav   No. Observations:               614559
Model:                            GLM   Df Residuals:                   614557
Model Family:                 Poisson   Df Model:                            1
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -8.5176e+05
Date:                Tue, 17 Jun 2025   Deviance:                   2.0972e+05
Time:                        17:56:46   Pearson chi2:                 2.18e+05
No. Iterations:                     4   Pseudo R-squ. (CS):           0.001764
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.6377      0.002    319.271      0.0

In [16]:
# Anova pour tester l'effet de l'heure sur la gravité de l'accident
# Important : Install statsmodels !

# Convertir la colonne 'hrmn' en type datetime
accidents_clean['hrmn'] = pd.to_datetime(accidents_clean['hrmn'], format='%H:%M')

# Extraire les heures et les minutes
hours = accidents_clean['hrmn'].dt.hour
minutes = accidents_clean['hrmn'].dt.minute

# Convertir en nombre décimal
accidents_clean['hrmn_continuous'] = hours + minutes / 60

import statsmodels.api as sm
import statsmodels.formula.api as smf

result = smf.ols('grav ~ hrmn_continuous', data= accidents_clean).fit()

sm.stats.anova_lm(result)

,df,sum_sq,mean_sq,F,PR(>F)
hrmn_continuous,1.0,370.365636,370.365636,582.05633,1.537482e-128
Residual,614557.0,391045.990281,0.636305,NaN,NaN


In [10]:
# GLM avec distribution de Poisson pour tester l'effet de l'age et la gravité sur le nombre d'accidents
# Important : Install statsmodels !

import statsmodels.api as sm
import statsmodels.formula.api as smf

accidents_age = accidents_clean.groupby(['age', 'grav']).size().reset_index(name='accident_count')

# Ajustement du GLM avec une distribution de Poisson
# La formule spécifie que 'accident_count' est modélisé en fonction de 'age' et 'grav'
glm_model = smf.glm(formula='accident_count ~ age * grav',
                    data=accidents_age,
                    family=sm.families.Poisson()).fit()

# Afficher le résumé du modèle
print(glm_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:         accident_count   No. Observations:                  411
Model:                            GLM   Df Residuals:                      407
Model Family:                 Poisson   Df Model:                            3
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -2.1876e+05
Date:                Tue, 17 Jun 2025   Deviance:                   4.3420e+05
Time:                        15:05:39   Pearson chi2:                 4.04e+05
No. Iterations:                     6   Pseudo R-squ. (CS):              1.000
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      9.3282      0.005   2032.077      0.0

In [13]:
import pandas as pd
import numpy as np
from itertools import combinations
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    if confusion_matrix.shape[0] == 1 or confusion_matrix.shape[1] == 1:
        return np.nan  # Pas assez de variabilité
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    return np.sqrt(phi2 / min(k - 1, r - 1))

def analyser_categorielle_redundance(df, cat_vars, target):
    print("\n=== Association entre chaque variable catégorielle et la target ===")
    cramer_target = {}
    for var in cat_vars:
        v = cramers_v(df[var], df[target])
        cramer_target[var] = v
        print(f"Cramér’s V ({var} vs {target}) = {v:.3f}")

    print("\n=== Détection des redondances entre variables catégorielles ===")
    redondantes = []
    for var1, var2 in combinations(cat_vars, 2):
        v = cramers_v(df[var1], df[var2])
        if v > 0.5:  # Seuil à ajuster selon ton cas
            redondantes.append((var1, var2, v))
            print(f"⚠️ {var1} et {var2} sont très corrélées (Cramér’s V = {v:.3f})")

    if not redondantes:
        print("✅ Aucune paire fortement redondante trouvée.")

    return cramer_target, redondantes

In [14]:


cat_vars = ['catr', 'agg', 'col', 'obs','obsm','choc','manv','motor','place','catu','grav','sexe','trajet','secu1','dep','com','situ','vma','catv_regroupée']
target = 'grav'

analyser_categorielle_redundance(accidents_clean, cat_vars, target)


=== Association entre chaque variable catégorielle et la target ===
Cramér’s V (catr vs grav) = 0.108
Cramér’s V (agg vs grav) = 0.169
Cramér’s V (col vs grav) = 0.152
Cramér’s V (obs vs grav) = 0.160
Cramér’s V (obsm vs grav) = 0.145
Cramér’s V (choc vs grav) = 0.123
Cramér’s V (manv vs grav) = 0.163
Cramér’s V (motor vs grav) = 0.102
Cramér’s V (place vs grav) = 0.145
Cramér’s V (catu vs grav) = 0.173
Cramér’s V (grav vs grav) = 1.000
Cramér’s V (sexe vs grav) = 0.091
Cramér’s V (trajet vs grav) = 0.110
Cramér’s V (secu1 vs grav) = 0.271
Cramér’s V (dep vs grav) = 0.162
Cramér’s V (com vs grav) = 0.345
Cramér’s V (situ vs grav) = 0.131
Cramér’s V (vma vs grav) = 0.132
Cramér’s V (catv_regroupée vs grav) = 0.235

=== Détection des redondances entre variables catégorielles ===
⚠️ catr et agg sont très corrélées (Cramér’s V = 0.638)
⚠️ catr et com sont très corrélées (Cramér’s V = 0.530)
⚠️ agg et com sont très corrélées (Cramér’s V = 0.710)
⚠️ agg et vma sont très corrélées (Cramér’s 

({'catr': 0.10817823609051375,
  'agg': 0.16917623412172333,
  'col': 0.1521256305144429,
  'obs': 0.15972847114912153,
  'obsm': 0.14476099415191693,
  'choc': 0.12266885411497513,
  'manv': 0.16298064318752525,
  'motor': 0.10150954866135381,
  'place': 0.1447105501758004,
  'catu': 0.17311479682390404,
  'grav': 1.0,
  'sexe': 0.09147605593349231,
  'trajet': 0.10962034052388471,
  'secu1': 0.2709434775196114,
  'dep': 0.16191855072621214,
  'com': 0.34546798096607867,
  'situ': 0.13120840242608864,
  'vma': 0.13188928075879522,
  'catv_regroupée': 0.23537934136140934},
 [('catr', 'agg', 0.6378699325025724),
  ('catr', 'com', 0.530182126828249),
  ('agg', 'com', 0.710123094075433),
  ('agg', 'vma', 0.8503861961914926),
  ('place', 'catu', 0.9964187699131044),
  ('catu', 'secu1', 0.5365521667326909),
  ('dep', 'com', 0.9689577470818683),
  ('com', 'vma', 0.5891635850989145)])